In [0]:

import requests
import re
import json
from datetime import datetime, timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

base_url = "https://hacker-news.firebaseio.com/v0"

session = requests.Session()

retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

def get_json(endpoint):
    response = session.get(
        f"{base_url}/{endpoint}.json",
        timeout=30
    )
    response.raise_for_status()
    return response.json()


In [0]:

user = get_json("user/whoishiring")

submitted_ids = sorted(
    user.get("submitted", []),
    reverse=True
)[:45]

hiring_threads = []

for item_id in submitted_ids:
    item = get_json(f"item/{item_id}")

    if (
        item
        and item.get("type") == "story"
        and re.search(
            r"who is hiring\?",
            item.get("title", ""),
            re.IGNORECASE
        )
    ):
        hiring_threads.append(item)

# Die zwei neuesten Hiring-Threads
threads = sorted(
    hiring_threads,
    key=lambda x: x["time"],
    reverse=True
)[:2]

if len(threads) < 2:
    raise RuntimeError(
        "Es wurden weniger als zwei Hiring-Threads gefunden."
    )

for thread in threads:
    print(
        thread["title"],
        "| ID:", thread["id"],
        "| Kommentare:", len(thread.get("kids", []))
    )


In [0]:

from pyspark.sql import functions as F

bronze_table = "job_market.bronze.hacker_news_api_raw"

thread_ids = [t["id"] for t in threads]

existing_df = (
    spark.table(bronze_table)
    .filter(F.col("thread_id").isin(thread_ids))
    .select(F.col("item.id").alias("post_id"))
    .distinct()
)

existing_ids = {
    row.post_id
    for row in existing_df.collect()
}

print("Bereits in Bronze:", len(existing_ids))


In [0]:

new_records = []
fetched_at = datetime.now(timezone.utc).isoformat()

for thread in threads:

    thread_id = thread["id"]

    thread_month = datetime.fromtimestamp(
        thread["time"],
        timezone.utc
    ).strftime("%Y-%m")

    post_ids = thread.get("kids", [])

    new_ids = [
        post_id
        for post_id in post_ids
        if post_id not in existing_ids
    ]

    print(
        thread_month,
        "| Neue Kommentare:", len(new_ids)
    )

    for post_id in new_ids:

        post = get_json(f"item/{post_id}")

        if post is None:
            raise RuntimeError(
                f"API lieferte kein Item für {post_id}"
            )

        if (
            post.get("id") != post_id
            or post.get("parent") != thread_id
        ):
            raise ValueError(
                f"Ungültige Zuordnung: {post_id}"
            )

        new_records.append({
            "source": "hacker_news_api",
            "thread_id": thread_id,
            "thread_month": thread_month,
            "fetched_at": fetched_at,
            "item": post
        })

print("Neue Datensätze insgesamt:", len(new_records))


In [0]:

import os
import uuid

volume_root = "/Volumes/job_market/landing/raw_files"

landing_dir = f"{volume_root}/hacker_news_api"
staging_dir = f"{volume_root}/_api_staging"

if new_records:

    dbutils.fs.mkdirs(landing_dir)
    dbutils.fs.mkdirs(staging_dir)

    run_id = datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%S")

    filename = (
        f"hn_api_{run_id}_{uuid.uuid4().hex[:8]}.jsonl"
    )

    staging_file = f"{staging_dir}/{filename}"
    final_file = f"{landing_dir}/{filename}"

    with open(staging_file, "w", encoding="utf-8") as f:
        for record in new_records:
            f.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )

    dbutils.fs.mv(staging_file, final_file)

    print("Neue Datei:", final_file)
    print("Datensätze:", len(new_records))

else:
    print("Keine neuen Kommentare – keine Datei erstellt.")


In [0]:

new_posts_count = len(new_records)

dbutils.jobs.taskValues.set(
    key="new_posts",
    value=new_posts_count
)

print(f"Neue Beiträge: {new_posts_count}")
